<h> <div align=center> <b> ĐỒ ÁN CUỐI KỲ </b> </div> </h>
<h> <div align=center> <b> BẢO MẬT TÍNH RIÊNG TƯ CHO MÔ HÌNH HỒI QUY TUYẾN TÍNH </b> </div> </h>
<h> <div align=center> <b> -------------------------------------------------------- </b> </div> </h>

In [ ]:
# Group 03
# 22120211 - Quach Ngoc Minh
# 22120226 - Le Trong Nghia
# 22120228 - Nguyen Minh Nghia

<h2> SETUP MÔI TRƯỜNG </h2>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import zscore

<h2> BÀI 1: TẬP DỮ LIỆU VỀ XE </h2>


In [ ]:
def load_data(train_path='train.csv', test_path='test.csv'):
    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)
    print("Train data info:")
    print(train_data.info())
    print("\nFirst 5 rows of train data:")
    print(train_data.head())
    return train_data, test_data

In [ ]:
train_file = 'data/train.csv'
test_file = 'data/test.csv'
train_data, test_data = load_data()

<h2> BÀI 2: TIỀN XỬ LÝ DỮ LIỆU

<h3> Hàm bổ trợ xử lý dữ liệu </h3>

In [ ]:
def manual_label_encode(series):
    classes = {label: idx for idx, label in enumerate(series.dropna().unique())}
    return series.map(classes), classes

Hàm **`manual_label_encode`** dùng để mã hóa thủ công các giá trị phân loại (categorical) trong một pandas.<br>
Series thành các số nguyên liên tiếp bắt đầu từ 0. <br>

- Hàm loại bỏ giá trị **`NaN`** trước khi thực hiện ánh xạ.

- Kết quả trả về là một cặp gồm:

 - Series mới đã mã hóa.

 - Từ điển ánh xạ giữa giá trị gốc và số nguyên tương ứng.

In [ ]:
def standardize(X):
    """Standardize data to normal distribution (mean=0, std=1)"""
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    return (X - mean) / std

Hàm **`standardize`** thực hiện chuẩn hóa dữ liệu: đưa mỗi cột (đặc trưng) của mảng đầu vào **`X`** về phân phối chuẩn với trung bình = 0 và độ lệch chuẩn = 1.


In [ ]:
def correlation_scores(X, y):
    """Calculate Pearson correlation coefficient between each column in X and y"""
    corr = []
    for i in range(X.shape[1]):
        xi = X[:, i]
        r = np.corrcoef(xi, y)[0, 1]
        corr.append(r)
    return np.array(corr)

Hàm **`correlation_scores`** tính hệ số tương quan Pearson giữa mỗi cột trong ma trận **`X`** với biến mục tiêu **`y`**.

Kết quả là một mảng chứa các hệ số tương quan, cho biết mức độ tuyến tính giữa từng đặc trưng và biến mục tiêu.

=> Hữu ích cho việc lựa chọn đặc trưng (feature selection).

In [ ]:
def extract_value_and_rpm(val):
    """Extract value and RPM from string."""
    try:
        parts = re.findall(r"(\d+\.?\d*)", str(val))
        if len(parts) >= 2:
            return float(parts[0]), float(parts[1])
        elif len(parts) == 1:
            return float(parts[0]), np.nan
        else:
            return np.nan, np.nan
    except:
        return np.nan, np.nan

Hàm **`extract_value_and_rpm`** dùng để trích xuất giá trị số từ một chuỗi văn bản — thường là giá trị công suất và tốc độ vòng quay (RPM).

Sử dụng biểu thức chính quy để tìm các số thực trong chuỗi.

Nếu tìm được 2 số: trả về cặp **`(value, RPM)`**. Nếu chỉ 1 số: trả về **`(value, NaN)`**. Nếu không tìm được: trả về **`(NaN, NaN)`**.

In [ ]:
def anova_f_test_score(X_cat, y):
    # X_cat: array of labels
    # y: continuous values (Price)
    groups = {}
    for xi, yi in zip(X_cat, y):
        groups.setdefault(xi, []).append(yi)
    group_means = {k: np.mean(v) for k, v in groups.items()}
    group_sizes = {k: len(v) for k, v in groups.items()}
    overall_mean = np.mean(y)
    ss_between = sum(
        group_sizes[k] * (group_means[k] - overall_mean) ** 2
        for k in groups
    )
    ss_within = sum(
        sum((yi - group_means[k]) ** 2 for yi in v)
        for k, v in groups.items()
    )
    df_between = len(groups) - 1
    df_within = len(y) - len(groups)
    ms_between = ss_between / df_between
    ms_within = ss_within / df_within
    f_score = ms_between / ms_within
    return f_score

Hàm **`anova_f_test_score`** tính toán chỉ số thống kê F (F-score) từ kiểm định ANOVA một chiều giữa một đặc trưng phân loại **`X_cat`** và biến mục tiêu liên tục **`y`**.

F-score cao thể hiện sự khác biệt lớn về giá trị trung bình **`y`** giữa các nhóm trong **`X_cat`**.

Thường dùng để đánh giá ý nghĩa của các đặc trưng rời rạc.

In [ ]:
def anova_f_test_onehot(X, y):
    """
    Calculate F-score for one-hot encoded columns in X based on y.
    - X: DataFrame with one-hot encoded columns
    - y: Series or array of continuous values (target)
    Returns:
        Series of F-scores for columns, sorted descending
    """
    from numpy import mean, sum as npsum
    scores = {}
    y_mean = mean(y)
    for col in X.columns:
        if set(X[col].unique()) <= {0, 1}:
            group1 = y[X[col] == 1]
            group0 = y[X[col] == 0]
            if len(group1) < 2 or len(group0) < 2: continue
            n1, n0 = len(group1), len(group0)
            mu1, mu0 = group1.mean(), group0.mean()
            ss_between = n1 * (mu1 - y_mean)**2 + n0 * (mu0 - y_mean)**2
            ss_within = npsum((group1 - mu1)**2) + npsum((group0 - mu0)**2)
            f_score = ss_between / (ss_within / (n1 + n0 - 2))
            scores[col] = f_score
    return pd.Series(scores).sort_values(ascending=False)

Hàm **`anova_f_test_onehot`** áp dụng kiểm định F cho từng cột one-hot trong bảng X.

Mỗi cột đại diện cho một nhãn phân loại nhị phân (0/1).

Hàm chia y thành hai nhóm tương ứng và tính toán F-score thể hiện sự khác biệt về trung bình giữa hai nhóm đó.

Kết quả là một **`pd.Series`** các F-score đã được sắp xếp giảm dần để chọn đặc trưng quan trọng.

<h3> Hàm tiền xử lý dữ liệu tập train.csv </h3>

In [ ]:
def preprocess_train_data(file_path):
    """
    Preprocess training data: handle missing values, encode features, remove outliers,
    standardize numeric columns, and select features.

    Args:
        file_path (str): Path to the training CSV file.

    Returns:
        tuple: (X, train_data, numeric_cols, freq_maps, train_mean, train_std, selected_features, owner_map)
    """
    # Load data
    train_data = pd.read_csv(file_path)
    df_missing = train_data[train_data.isna().any(axis=1)]

    # Extract values from Max Power, Max Torque, and Engine
    train_data[['Max_Power_Value', 'Max_Power_RPM']] = train_data['Max Power'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
    train_data[['Max_Torque_Value', 'Max_Torque_RPM']] = train_data['Max Torque'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
    train_data['Engine'] = train_data['Engine'].str.replace('cc', '').astype(float)

    # Define numeric and categorical columns
    numeric_cols = ['Year', 'Kilometer', 'Engine', 'Length', 'Width', 'Height', 'Seating Capacity', 'Fuel Tank Capacity', 'Max_Power_Value', 'Max_Power_RPM', 'Max_Torque_Value', 'Max_Torque_RPM']
    categorical_cols = ['Make', 'Location', 'Fuel Type', 'Transmission', 'Seller Type', 'Owner', 'Drivetrain']
    train_data_encoded = train_data.copy()

    # Remove rows with missing values
    mask = train_data.drop(columns=['Price']).notna().all(axis=1) & train_data['Price'].notna()
    train_data = train_data[mask].reset_index(drop=True)

    # Remove outliers based on Z-score
    z_scores = zscore(train_data['Price'])
    mask_outlier = np.abs(z_scores) < 3
    train_data = train_data[mask_outlier].reset_index(drop=True)

    # Plot histogram of Price
    plt.hist(train_data['Price'], bins=50)
    plt.title("Price Distribution After Outlier Removal")
    plt.show()

    # Label encode Owner
    owner_order = ['First', 'Second', 'Third', 'Fourth', '4 or More', 'UnRegistered Car']
    owner_map = {v: i for i, v in enumerate(owner_order)}
    train_data['Owner'] = train_data['Owner'].map(owner_map)

    # Frequency encoding for Make and Location
    freq_maps = {}
    for col in ['Make', 'Location']:
        freq_map = train_data[col].value_counts(normalize=True).to_dict()
        freq_maps[col] = freq_map
        train_data[col + '_Freq'] = train_data[col].map(freq_map)

    # One-hot encoding for nominal columns
    one_hot_cols = ['Fuel Type', 'Transmission', 'Seller Type', 'Drivetrain']
    train_data = pd.get_dummies(train_data, columns=one_hot_cols)

    # Calculate ANOVA F-scores
    anova_scores = anova_f_test_onehot(train_data, train_data['Price'])
    owner_fscore = anova_f_test_score(train_data['Owner'].to_numpy(), train_data['Price'])
    anova_scores = pd.concat([anova_scores, pd.Series({'Owner': owner_fscore})])

    onehot_scores = pd.DataFrame({
        'Feature': anova_scores.index,
        'ANOVA_F_Score': anova_scores.values
    }).sort_values(by='ANOVA_F_Score', ascending=False)

    print("ANOVA F-Scores:\n", onehot_scores)

    # Calculate correlation scores
    y = train_data['Price'].to_numpy()
    x = train_data[numeric_cols].to_numpy()
    drop_cols = ['Model', 'Max Power', 'Max Torque', 'Color']
    train_data = train_data.drop(columns=drop_cols, axis=1)
    x_scaled = standardize(x)
    correlations = correlation_scores(x_scaled, y)
    correlations_freq = correlation_scores(train_data[['Make_Freq','Location_Freq']].to_numpy(), y)

    feature_scores = pd.DataFrame({
        'Feature': numeric_cols + ['Make_Freq', 'Location_Freq'],
        'Correlation_with_Price': np.concatenate((correlations, correlations_freq))
    }).sort_values(by='Correlation_with_Price', ascending=False)

    print("Correlation Scores:\n", feature_scores)

    # Drop unused columns
    # drop_cols = ['Model', 'Max Power', 'Max Torque', 'Color']
    # train_data = train_data.drop(columns=drop_cols, axis=1)
    # train_data.to_csv('train_data_preprocessed_new.csv', index=False)

    # Standardize numeric columns
    train_mean = np.mean(train_data[numeric_cols].to_numpy(), axis=0)
    train_std = np.std(train_data[numeric_cols].to_numpy(), axis=0)
    train_data[numeric_cols] = standardize(train_data[numeric_cols].to_numpy())

    # Select features
    selected_features = feature_scores[np.abs(feature_scores['Correlation_with_Price']) > 0.5]['Feature'].tolist()
    selected_features += ['Year']
    selected_features += onehot_scores[onehot_scores['ANOVA_F_Score'] > 120]['Feature'].tolist()
    print("Selected features:", selected_features)
    print("Number of selected features:", len(selected_features))
    X = train_data[selected_features].copy()
    return X, train_data, numeric_cols, freq_maps, train_mean, train_std, selected_features, owner_map

Hàm **`preprocess_train_data`** thực hiện tiền xử lý dữ liệu huấn luyện từ file CSV, bao gồm các bước sau:
1. **Đọc dữ liệu**
 - Đọc dữ liệu từ **`file_path`** sử dụng **`pd.read_csv`**.

2. **Trích xuất thông tin số từ cột văn bản**
 - Các cột: **`Max Power`**, **`Max Torque`**, và **`Engine`** chứa văn bản như **`"110 PS @ 5000 rpm"`**.
 - Trích xuất giá trị số và chuyển thành các cột mới: **`Max_Power_Value`**,**`Max_Power_RPM`**; **`Max_Torque_Value`**, **`Max_Torque_RPM`**; **`Engine`**: loại bỏ **`'cc'`** và chuyển về kiểu **`float`**
3. **Phân loại cột**
 - Cột số: **`numeric_cols`** — các đặc trưng dạng số liên tục.
 - Cột phân loại: **`categorical_cols`** — các đặc trưng dạng phân loại.
4. **Xử lý giá trị thiếu (`NaN`)**
 - Loại bỏ các hàng chứa giá trị thiếu bất kỳ (trừ **`Price`** nếu cần).
5. **Loại bỏ outlier (`Price`)**
 - Sử dụng Z-score để phát hiện và loại bỏ ngoại lệ trong cột **`Price`**.
6. **Hiển thị biểu đồ phân phối `Price`**
 - Vẽ biểu đồ histogram để trực quan hóa phân phối **`Price`** sau khi loại bỏ ngoại lai.
7. **Label Encoding cho `Owner`**
 - Chuyển đổi các giá trị phân loại có thứ tự trong **`Owner`** sang số nguyên theo thứ tự:
  - **`First`** → 0, **`Second`** → 1, ..., **`UnRegistered Car`** → 5
8. **Frequency Encoding cho `Make` và `Location`**
 - Mỗi nhãn được mã hóa bằng tần suất xuất hiện của nó trong dữ liệu.
9. **One-hot Encoding cho các cột phân loại còn lại**
 - Gồm: **`Fuel Type`**, **`Transmission`**, **`Seller Type`**, **`Drivetrain`**.
10. **Tính toán chỉ số ANOVA F-score**
 - Đánh giá mức độ liên quan giữa các đặc trưng phân loại (đặc biệt là one-hot) và giá trị **`Price`**.
11. **Tính hệ số tương quan Pearson**
 - Tính mức tương quan tuyến tính giữa **`Price`** và:
     - Các đặc trưng số (**`numeric_cols`**)
     - Các đặc trưng đã frequency encoded (**`Make_Freq`**, **`Location_Freq`**)
12. **Chuẩn hóa (standardize) các cột số**
 - Đưa các đặc trưng số về phân phối chuẩn với:
     - Trung bình = 0
     - Độ lệch chuẩn = 1
13. **Chọn đặc trưng đầu vào (`selected_features`)**
 - Các đặc trưng được chọn nếu:
     - **Hệ số tương quan > 0.5**
     - **ANOVA F-score > 120**
 - Luôn bao gồm **`Year`** trong danh sách.
14. **Trả về kết quả**
 - **`X`**: Dữ liệu đầu vào đã chọn đặc trưng.
 - **`train_data`**: Dữ liệu đầy đủ sau xử lý.
 - **`numeric_cols`**: Danh sách cột số ban đầu.
 - **`freq_maps`**: Bản đồ frequency encoding cho **`Make`** và **`Location`**.
 - **`train_mean`**, **`train_std`**: Giá trị thống kê chuẩn hóa các đặc trưng số.
 - **`selected_features`**: Danh sách đặc trưng được chọn để đưa vào mô hình.
 - **`owner_map`**: Bản đồ label encoding cho **`Owner`**.


Nhóm em quyết định lựa chọn các đặc trưng thỏa mãn độ lớn hệ số tương quan **> 0.5** và kiểm tra **ANOVA > 120**. <br>
Ngoài ra nhóm cũng lựa chọn thêm đặc trưng **`Year`** bởi vì kiến thức phổ thông rằng xe đời cũ thì thường giá sẽ thấp hơn

=> Các đặc trưng được lựa chọn: [**`Max_Power_Value`**, **`Max_Torque_Value`**, **`Length`**, **`Fuel Tank Capacity`**, **`Width`**, **`Engine`**, **`Year`**, **`Drivetrain_FWD`**, **`Transmission_Automatic`**, **`Transmission_Manual`**, **`Drivetrain_AWD`**, **`Drivetrain_RWD`**]

<h3> Hàm tiền xử lý dữ liệu tập test.csv </h3>

In [ ]:
def preprocess_test_data(file_path, train_data, numeric_cols, freq_maps,
                         train_mean, train_std, selected_features, owner_map):
    """
    Preprocess the test dataset: handle missing values, extract numeric info,
    encode categorical features, normalize numeric columns using training stats,
    and select final features.

    Args:
        file_path (str): Path to the test CSV file.
        train_data (pd.DataFrame): Preprocessed training DataFrame.
        numeric_cols (list): List of numeric column names.
        freq_maps (dict): Frequency maps for 'Make' and 'Location' from training data.
        train_mean (np.ndarray): Mean values for numeric columns in training data.
        train_std (np.ndarray): Standard deviation for numeric columns in training data.
        selected_features (list): Final list of selected feature column names.
        owner_map (dict): Mapping of Owner string labels to integers.

    Returns:
        tuple: (x_test, y_test, full_preprocessed_test_data)
    """

    # Load test data
    test_data = pd.read_csv(file_path)

    # Extract numeric values from 'Max Power' and 'Max Torque' columns
    test_data[['Max_Power_Value', 'Max_Power_RPM']] = test_data['Max Power'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
    test_data[['Max_Torque_Value', 'Max_Torque_RPM']] = test_data['Max Torque'].apply(lambda x: pd.Series(extract_value_and_rpm(x)))
    test_data['Engine'] = test_data['Engine'].str.replace('cc', '').astype(float)

    mask = test_data.drop(columns=['Price']).notna().all(axis=1) & test_data['Price'].notna()
    test_data = test_data[mask].reset_index(drop=True)

    # Encode Owner
    owner_order = ['First', 'Second', 'Third', 'Fourth', '4 or More', 'UnRegistered Car']
    owner_map = {v: i for i, v in enumerate(owner_order)}
    test_data['Owner'] = test_data['Owner'].map(owner_map)

    # Frequency encoding for 'Make' and 'Location' based on training stats
    for col in ['Make', 'Location']:
        freq_map = train_data[col].value_counts(normalize=True).to_dict()
        test_data[col + '_Freq'] = test_data[col].map(freq_map)

    # One-hot encode nominal categorical columns
    one_hot_cols = ['Fuel Type', 'Transmission', 'Seller Type', 'Drivetrain']
    test_data = pd.get_dummies(test_data, columns=one_hot_cols)

    # Drop unnecessary columns if present
    drop_cols = ['Model', 'Max Power', 'Max Torque', 'Color']
    test_data = test_data.drop(columns=drop_cols, axis=1)

    # Normalize numeric columns using training data statistics
    test_data[numeric_cols] = (test_data[numeric_cols] - train_mean) / train_std

    # Select only the features used in the model
    x_test = test_data[selected_features]
    # Extract target variable
    y_test = test_data['Price'].to_numpy()

    return x_test, y_test, test_data

1. **Đọc dữ liệu** từ đường dẫn **`file_path`**.
2. **Trích xuất thông tin số từ chuỗi văn bản:**
 - **`Max Power`** và **`Max Torque`**: tách giá trị và RPM.
 - **`Engine`**: chuyển đổi chuỗi **`1234cc`** thành số thực **`1234.0`**.
3. **Xử lý giá trị thiếu** (**`NaN`**):
 - Loại bỏ hàng có bất kỳ cột nào bị thiếu (bao gồm **`Price`**).
4. **Label Encoding** cho **`Owner`**:
 - Ánh xạ theo **`owner_map`** lấy từ tập huấn luyện.
 - Gán **`0`** cho giá trị chưa từng thấy.
5. **Tần suất hóa** (**Frequency Encoding**):
 - **`Make`**, **`Location`**: ánh xạ theo **`freq_maps`** từ tập huấn luyện.
 - Gán **`0`** nếu giá trị chưa từng xuất hiện.
6. **One-hot Encoding** cho các cột phân loại: **`Fuel Type`**, **`Transmission`**, **`Seller Type`**, **`Drivetrain`**.
7. **Căn chỉnh các cột one-hot với tập huấn luyện:**
 - Thêm cột còn thiếu với giá trị **`0`**.
 - Giữ lại các cột giao nhau giữa test và train.
8. **Loại bỏ các cột không dùng** nếu chúng tồn tại: **`Model`**, **`Max Power`**, **`Max Torque`**, **`Color`**.
9. **Chuẩn hóa các cột số** (**`numeric_cols`**) theo:
 - **`train_mean`** và **`train_std`** từ tập huấn luyện.
10. **Chọn đặc trưng đầu vào** (**`selected_features`**) đã xác định từ huấn luyện.
11. **Trả về**:
 - **`x_test`**: đầu vào cho mô hình.
 - **`y_test`**: nhãn thật (**`Price`**).
 - **`test_data`**: toàn bộ DataFrame sau khi xử lý.

In [ ]:
X_train, train_data, numeric_cols, freq_maps, train_mean, train_std, selected_features, owner_map = preprocess_train_data(train_file)
x_test, y_test, test_data = preprocess_test_data(test_file, train_data, numeric_cols, freq_maps, train_mean, train_std, selected_features, owner_map)

<h2> BÀI 3: MÔ HÌNH TUYẾN TÍNH VÀ ỨNG DỤNG DP </h2>

<h3> Lớp Hồi quy Tuyến tính </h3>

In [ ]:
class LinearRegression():
    def __init__(self, alpha=0.11, l2_lambda=0.1, num_iterations=1000,
                 transform_func=None, dp_epsilon=None, dp_delta=None, dp_sensitivity=None):
        """
        Linear Regression model with optional L2 regularization and Differential Privacy.

        Args:
            alpha (float): Learning rate.
            l2_lambda (float): Regularization strength (L2).
            num_iterations (int): Number of training iterations.
            transform_func (callable): Function to transform features (e.g., polynomial).
            dp_epsilon (float): Privacy budget for Differential Privacy (if used).
            dp_delta (float): Probability bound for Differential Privacy (if used).
            dp_sensitivity (float): Sensitivity for DP noise calculation (if used).
        """
        self.alpha = alpha
        self.l2_lambda = l2_lambda
        self.num_iterations = num_iterations
        self.transform_func = transform_func
        self.weights = None
        self.bias = 0

        # Differential Privacy parameters
        self.dp_epsilon = dp_epsilon
        self.dp_delta = dp_delta
        self.dp_sensitivity = dp_sensitivity

    def fit(self, X, y):
        """
        Train the model on input features X and targets y.

        Args:
            X (pd.DataFrame or np.ndarray): Input feature matrix.
            y (np.ndarray): Target values (regression).
        """
        X_transformed = self.transform_func(X)
        n_samples, n_features = X_transformed.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        # If DP is enabled, calculate Gaussian noise
        if self.dp_epsilon and self.dp_sensitivity and self.dp_delta:
            sigma = self.dp_sensitivity * np.sqrt(2 * np.log(1.25 / self.dp_delta)) / self.dp_epsilon
            noise = np.random.normal(0, sigma, size=n_features)
        else:
            noise = 0

        # Gradient Descent loop
        for _ in range(self.num_iterations):
            predictions = np.dot(X_transformed, self.weights) + self.bias
            errors = predictions - y

            # Compute gradients
            gradient_weights = (np.dot(X_transformed.T, errors) + 2 * self.l2_lambda * self.weights) / n_samples
            gradient_bias = np.mean(errors)

            # Update parameters
            self.weights -= self.alpha * gradient_weights
            self.bias -= self.alpha * gradient_bias

        # Add noise to weights after training (for DP)
        if isinstance(noise, np.ndarray):
            self.weights += noise

    def predict(self, X):
        """
        Make predictions using the trained model.

        Args:
            X (pd.DataFrame or np.ndarray): Input features.

        Returns:
            np.ndarray: Predicted values.
        """
        X_transformed = self.transform_func(X)
        return np.dot(X_transformed, self.weights) + self.bias

    def get_weights(self):
        """
        Get the model's current weights and bias.

        Returns:
            tuple: (weights, bias)
        """
        return self.weights, self.bias

<h3> Hàm bổ trợ mô hình tuyến tính </h3>

In [ ]:
def make_transform_func_exact_degree(degree_dict):
    """
    Create a transformation function based on exact polynomial degrees for selected features.

    Args:
        degree_dict (dict): A dictionary specifying how to generate features.
            - For a single column: {'col_name': [1, 2]} → include x, x^2
            - For sum of columns: {('sum', ('col1', 'col2')): [1]} → include (col1 + col2)^1
            - For product of columns: {('prod', ('col1', 'col2')): [1, 2]} → include col1*col2, (col1*col2)^2

    Returns:
        function: A transformation function that takes a DataFrame X and returns a NumPy array
                  with the generated polynomial features.
    """
    def transform_func(X):
        if isinstance(X, pd.Series):
            X = X.to_frame().T
        elif not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        transformed_features = []

        for key, degrees in degree_dict.items():
            if isinstance(key, str):
                # Single column feature transformation
                x_col = X[key].astype(float).to_numpy().reshape(-1, 1)
                transformed_features.append(np.hstack([x_col ** d for d in degrees]))

            elif isinstance(key, tuple) and len(key) == 2:
                operation, columns = key
                columns = list(columns)
                col_arrays = [X[col].astype(float).to_numpy().reshape(-1, 1) for col in columns]

                if operation == "sum":
                    base = np.sum(col_arrays, axis=0)
                elif operation == "prod":
                    base = np.prod(col_arrays, axis=0)
                else:
                    raise ValueError(f"Unsupported operation: {operation}")

                transformed_features.append(np.hstack([base ** d for d in degrees]))

            else:
                raise ValueError(f"Invalid key format: {key}")

        return np.hstack(transformed_features) if transformed_features else np.empty((len(X), 0))

    return transform_func

In [ ]:
def evaluate_regression(model, x_test, y_test):
    """
    Evaluate a linear regression model.

    Returns:
        A dictionary containing MSE, RMSE, MAE, and adjusted R² score.
    """
    y_pred_log = model.predict(x_test)
    y_pred = np.expm1(y_pred_log)  # Convert back from log scale to original scale

    # Compute evaluation metrics
    mse = np.mean((y_pred - y_test) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_pred - y_test))

    # R² score
    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0

    # Adjusted R² score
    n = len(y_test)
    p = x_test.shape[1] if hasattr(x_test, "shape") else len(x_test[0])
    adj_r2 = 1 - ((1 - r2) * (n - 1)) / (n - p - 1) if (n - p - 1) != 0 else r2

    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': adj_r2
    }

<h3> Thực nghiệm mô hình hồi quy tuyến tính </h3>

1. $\hat{y} = w_0 + \sum_{i=1}^{12} w_i x_i$

In [ ]:
degree_dict_1 = {
    "Max_Power_Value": [1],
    "Max_Torque_Value": [1],
    "Length": [1],
    "Fuel Tank Capacity": [1],
    "Width": [1],
    "Engine": [1],
    "Year": [1],
    "Drivetrain_FWD": [1],
    "Transmission_Automatic": [1],
    "Transmission_Manual": [1],
    "Drivetrain_AWD": [1],
    "Drivetrain_RWD": [1]
}

# Create transformation function using exact polynomial degrees
predict_func = make_transform_func_exact_degree(degree_dict_1)

# Initialize the LinearRegression model with the transformation
model = LinearRegression(transform_func=predict_func)

# Apply log transform to the target variable
y = train_data['Price'].to_numpy()
y_log = np.log1p(y)

# Train the model
X = train_data[selected_features].copy()
model.fit(X, y_log)

# Evaluate the model on the test set
result_test = evaluate_regression(model, x_test, y_test)
print("Test set results for Model 1:", result_test)

# Evaluate the model on the training set
result_train = evaluate_regression(model, X, y)
print("Training set results for Model 1:", result_train)

2. $\hat{y} = w_0 + w_1 x_{\text{power}} + w_2 x_{\text{power}}^2 + w_3 x_{\text{torque}} + w_4 x_{\text{torque}}^2 + w_5 x_{\text{length}} + w_6 x_{\text{length}}^2 + w_7 x_{\text{fuel}} + w_8 x_{\text{width}} + w_9 x_{\text{engine}} + w_{10} x_{\text{year}} + w_{11} x_{\text{FWD}} + w_{12} x_{\text{auto}} + w_{13} x_{\text{manual}} + w_{14} x_{\text{AWD}} + w_{15} x_{\text{RWD}}$

In [ ]:
degree_dict_2 = {
    "Max_Power_Value": [1, 2],
    "Max_Torque_Value": [1, 2],
    "Length": [1, 2],
    "Fuel Tank Capacity": [1],
    "Width": [1],
    "Engine": [1],
    "Year": [1],
    "Drivetrain_FWD": [1],
    "Transmission_Automatic": [1],
    "Transmission_Manual": [1],
    "Drivetrain_AWD": [1],
    "Drivetrain_RWD": [1]
}

# Create a transform function with specific polynomial degrees
transform_func_2 = make_transform_func_exact_degree(degree_dict_2)

# Initialize and train the regression model
model_2 = LinearRegression(transform_func=transform_func_2)
model_2.fit(X, y_log)

# Print the model's weights
print("Weights of Model 2:", model_2.get_weights())

# Evaluate on the test set
result_test_2 = evaluate_regression(model_2, x_test, y_test)
print("Results on test set for Model 2:", result_test_2)

# Evaluate on the training set
result_train_2 = evaluate_regression(model_2, X, y)
print("Results on training set for Model 2:", result_train_2)

3. $\hat{y} = w_0 + w_1 x_{\text{power}} + w_2 x_{\text{power}}^2 + w_3 x_{\text{torque}} + w_4 x_{\text{length}} + w_5 x_{\text{fuel}} + w_6 x_{\text{width}} + w_7 x_{\text{width}}^2 + w_8 x_{\text{engine}} + w_9 x_{\text{year}} + w_{10} x_{\text{FWD}} + w_{11} x_{\text{auto}} + w_{12} x_{\text{manual}} + w_{13} x_{\text{AWD}} + w_{14} x_{\text{RWD}}$

In [ ]:
degree_dict_3 = {
    "Max_Power_Value": [1, 2],
    "Max_Torque_Value": [1],
    "Length": [1],
    "Fuel Tank Capacity": [1],
    "Width": [1, 2],
    "Engine": [1],
    "Year": [1],
    "Drivetrain_FWD": [1],
    "Transmission_Automatic": [1],
    "Transmission_Manual": [1],
    "Drivetrain_AWD": [1],
    "Drivetrain_RWD": [1]
}

# Create transformation function with specific polynomial degrees
transform_func_3 = make_transform_func_exact_degree(degree_dict_3)

# Initialize and train the model using transformed features
model_3 = LinearRegression(transform_func=transform_func_3)
model_3.fit(X, y_log)

# Output model weights
print("Weights of Model 3:", model_3.get_weights())

# Evaluate model on the test set
result_test_3 = evaluate_regression(model_3, x_test, y_test)
print("Test set results for Model 3:", result_test_3)

# Evaluate model on the training set
result_train_3 = evaluate_regression(model_3, X, y)
print("Training set results for Model 3:", result_train_3)

4. $\hat{y} = w_0 + w_1 x_{\text{power}} + w_2 x_{\text{power}}^2 + w_3 x_{\text{torque}} + w_4 x_{\text{torque}}^2 + w_5 x_{\text{length}} + w_6 x_{\text{length}}^2 + w_7 x_{\text{fuel}} + w_8 x_{\text{width}} + w_9 x_{\text{engine}} + w_{10} x_{\text{year}} + w_{11} x_{\text{FWD}} + w_{12} x_{\text{auto}} + w_{13} x_{\text{manual}} + w_{14} x_{\text{AWD}} + w_{15} x_{\text{RWD}} + w_{16} (x_{\text{width}} \times x_{\text{engine}})$

In [ ]:
degree_dict_4 = {
    "Max_Power_Value": [1, 2],
    "Max_Torque_Value": [1, 2],
    "Length": [1, 2],
    "Fuel Tank Capacity": [1],
    "Width": [1],
    "Engine": [1],
    "Year": [1],
    "Drivetrain_FWD": [1],
    "Transmission_Automatic": [1],
    "Transmission_Manual": [1],
    "Drivetrain_AWD": [1],
    "Drivetrain_RWD": [1],
    ('prod', ('Width', 'Engine')): [1],
}

# Create a transformation function with the selected polynomial degrees
transform_func_4 = make_transform_func_exact_degree(degree_dict_4)

# Initialize and train the linear regression model using the selected features
model_4 = LinearRegression(transform_func=transform_func_4)
model_4.fit(X, y_log)

# Print model weights
print("Weights of Model 4:", model_4.get_weights())

# Evaluate the model on the test set
result_test_4 = evaluate_regression(model_4, x_test, y_test)
print("Test set results for Model 4:", result_test_4)

# Evaluate the model on the training set
result_train_4 = evaluate_regression(model_4, X, y)
print("Training set results for Model 4:", result_train_4)

<h3> Cơ chế tạo nhiễu </h3>

In [ ]:
random_state = np.random.RandomState(42)

<h4> Cơ chế Gauss </h4>

In [ ]:
def gaussian_mechanism(preds, sensitivity, epsilon, delta):
    """
    Apply Gaussian noise to predictions to ensure (ε, δ)-Differential Privacy.

    Args:
        preds (np.ndarray): Predicted values (model output).
        sensitivity (float): L2 sensitivity of the output function.
        epsilon (float): Privacy budget (ε).
        delta (float): Privacy parameter (δ).

    Returns:
        np.ndarray: Predictions with added Gaussian noise.
    """
    sigma = sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    noise = random_state.normal(loc=0.0, scale=sigma, size=preds.shape)
    return preds + noise

<h4> Cơ chế Laplace </h4>

In [ ]:
def laplace_mechanism(preds, sensitivity, epsilon):
    """
    Apply Laplace noise to predictions to ensure ε-Differential Privacy.

    Args:
        preds (np.ndarray): Predicted values (model output).
        sensitivity (float): L1 sensitivity of the output function.
        epsilon (float): Privacy budget (ε).

    Returns:
        np.ndarray: Predictions with added Laplace noise.
    """
    b = sensitivity / epsilon
    noise = random_state.laplace(loc=0.0, scale=b, size=preds.shape)
    return preds + noise

<h4>  Định lý Tổ hợp Nâng cao (Advanced Composition Theorem) </h4>

In [ ]:
def advanced_comp_epsilon(epsilon, delta, k, delta_prime):
    """
    Compute the total (ε, δ) privacy guarantee after k compositions
    of an (ε, δ)-differentially private mechanism using the Advanced Composition Theorem.

    Args:
        epsilon (float): Privacy budget per iteration (ε).
        delta (float): Failure probability per iteration (δ).
        k (int): Number of iterations (compositions).
        delta_prime (float): Additional failure probability added in composition.

    Returns:
        tuple:
            eps_tot (float): Total composed epsilon.
            delta_tot (float): Total composed delta.
    """
    eps_tot = np.sqrt(2 * k * np.log(1 / delta_prime)) * epsilon + k * epsilon * (np.exp(epsilon) - 1)
    delta_tot = k * delta + delta_prime
    return eps_tot, delta_tot

<h4> Ước lượng độ nhạy L2 tối đa của các dự đoán </h4>

In [ ]:
def estimate_output_sensitivity(model_class, X_train, y_train, X_test, transform_func=None, sens_type=2):
    """
    Estimate the L2 sensitivity of predictions using leave-one-out method.

    Args:
        model_class (class): The regression model class (e.g., LinearRegression).
        X_train (pd.DataFrame): Training feature data.
        y_train (pd.Series or np.ndarray): Target values.
        X_test (pd.DataFrame): Feature data to compute prediction sensitivity on.
        transform_func (callable, optional): Optional feature transformation function.

    Returns:
        float: Maximum L2 sensitivity of predictions.
    """
    n = X_train.shape[0]

    # Fit the model on the full training set
    model_full = model_class(transform_func=transform_func)
    model_full.fit(X_train, y_train)
    full_preds = model_full.predict(X_test)

    max_diff = 0.0
    for i in range(n):
        # Leave-one-out training data
        X_leave = X_train.drop(X_train.index[i])
        y_leave = y_train.drop(y_train.index[i]) if isinstance(y_train, pd.Series) else np.delete(y_train, i)

        model_i = model_class(transform_func=transform_func)
        model_i.fit(X_leave, y_leave)
        preds_i = model_i.predict(X_test)

        # L2 norm of prediction difference
        diff = np.linalg.norm(full_preds - preds_i, ord=sens_type)
        max_diff = max(max_diff, diff)

    return max_diff

<h4> Thực nghiệm mô hình 3 có (ε, δ) - DP với cơ chế Gauss có các epsilon = [0.8, 8.0, 15.0], delta=1e-5, có nhiễu ngẫu nhiên </h4>

In [ ]:
model = LinearRegression
transform = transform_func_3
sens_type = 2 # L2 sensitivity
delta = 1e-5
eps_values = [0.8, 8.0, 15.0]

In [ ]:
# Get min and max of the transformed training labels (log1p)
y_train_log_min = np.log1p(y.min())
y_train_log_max = np.log1p(y.max())

# Estimate sensitivity of model output under transformation
sensitivity_pred = estimate_output_sensitivity(model, X, y_log, x_test, transform, sens_type=sens_type)

In [ ]:
results_dp = {}

# Evaluate DP mechanisms for different epsilon values
for eps in eps_values:
    y_pred_log = model_2.predict(x_test)
    y_pred_log_clipped = np.clip(y_pred_log, a_min=y_train_log_min, a_max=y_train_log_max)

    # Apply Gaussian mechanism
    y_pred_log_dp_gaussian = gaussian_mechanism(y_pred_log_clipped, sensitivity_pred, eps, delta)
    y_pred_log_dp_gaussian = np.clip(y_pred_log_dp_gaussian, a_min=y_train_log_min, a_max=y_train_log_max)
    y_dp_gaussian = np.expm1(y_pred_log_dp_gaussian)  # Inverse of log1p to get original scale

    # Apply Laplace mechanism
    y_pred_log_dp_laplace = laplace_mechanism(y_pred_log_clipped, sensitivity_pred, eps)
    y_pred_log_dp_laplace = np.clip(y_pred_log_dp_laplace, a_min=y_train_log_min, a_max=y_train_log_max)
    y_dp_laplace = np.expm1(y_pred_log_dp_laplace)

    # Evaluation metrics for Gaussian
    mse_g = np.mean((y_dp_gaussian - y_test) ** 2)
    rmse_g = np.sqrt(mse_g)
    mae_g = np.mean(np.abs(y_dp_gaussian - y_test))
    ss_res_g = np.sum((y_test - y_dp_gaussian) ** 2)
    r2_g = 1 - ss_res_g / np.sum((y_test - y_test.mean()) ** 2)

    # Evaluation metrics for Laplace
    mse_l = np.mean((y_dp_laplace - y_test) ** 2)
    rmse_l = np.sqrt(mse_l)
    mae_l = np.mean(np.abs(y_dp_laplace - y_test))
    ss_res_l = np.sum((y_test - y_dp_laplace) ** 2)
    r2_l = 1 - ss_res_l / np.sum((y_test - y_test.mean()) ** 2)

    # Advanced composition for total epsilon and delta (1 query)
    eps_tot, delta_tot = advanced_comp_epsilon(eps, delta, k=1, delta_prime=1e-6)

    results_dp[eps] = {
        'Gaussian': {'MSE': mse_g, 'RMSE': rmse_g, 'MAE': mae_g, 'R2': r2_g},
        'Laplace': {'MSE': mse_l, 'RMSE': rmse_l, 'MAE': mae_l, 'R2': r2_l},
        'eps_total': eps_tot, 'delta_total': delta_tot
    }

In [ ]:
print(f"y_train_log_min = {y_train_log_min}, y_train_log_max = {y_train_log_max}")
print(f"Estimated L2(L1)-sensitivity S (prediction) = {sensitivity_pred:.4f}")

for eps, stats in results_dp.items():
    print(f"\nε = {eps:.1f}  →  ε_total = {stats['eps_total']:.3f}, δ_total = {stats['delta_total']:.1e}")
    print(f"Gaussian: MSE = {stats['Gaussian']['MSE']:.2e}, RMSE = {stats['Gaussian']['RMSE']:.2f}, "
          f"MAE = {stats['Gaussian']['MAE']:.2f}, R2 = {stats['Gaussian']['R2']:.3f}")
    # print(f"Laplace:  MSE = {stats['Laplace']['MSE']:.2e}, RMSE = {stats['Laplace']['RMSE']:.2f}, "
    #       f"MAE = {stats['Laplace']['MAE']:.2f}, R2 = {stats['Laplace']['R2']:.3f}")

<h3> Mô hình Federated Learning với Gaussian Differential Privacy </h3>

Mô hình hồi quy tuyến tính phân tán với nhiễu Gaussian nhằm đảm bảo tính riêng tư vi phân. Mô hình sử dụng kỹ thuật cắt gradient theo mẫu và thêm nhiễu Gaussian, tuân theo nguyên tắc DP-SGD trong môi trường học máy phân tán.

In [ ]:
class FederatedGaussianDP:
    def __init__(self, clip_norm=1.0, noise_multiplier=1.0,
                 lr=0.01, local_epochs=1, batch_size=32):
        self.clip_norm = clip_norm
        self.noise_multiplier = noise_multiplier
        self.lr = lr
        self.local_epochs = local_epochs
        self.batch_size = batch_size
        self.weights = None
        self.bias = 0.0

    def _local_update(self, X, y):
        # X: np.ndarray (n_samples, n_features), y: np.ndarray
        n, d = X.shape
        w, b = self.weights.copy(), self.bias
        for _ in range(self.local_epochs):
            idx = np.random.permutation(n)
            for start in range(0, n, self.batch_size):
                end = start + self.batch_size
                batch_idx = idx[start:end]
                Xb, yb = X[batch_idx], y[batch_idx]

                grads = []
                preds = Xb.dot(w) + b
                errors = preds - yb
                for xi, ei in zip(Xb, errors):
                    gi = 2 * ei * xi
                    norm = np.linalg.norm(gi)
                    gi = gi * min(1, self.clip_norm / (norm + 1e-12))
                    grads.append(gi)
                # average clipped gradient
                g_local = np.mean(grads, axis=0)
                # bias gradient
                grad_b = np.mean(2 * errors)
                # return local update
                yield g_local, grad_b

    def federated_train(self, client_data, rounds=10):
        # client_data: list of (X_client, y_client)
        self.weights = np.zeros(client_data[0][0].shape[1])
        self.bias = 0.0

        for r in range(rounds):
            sum_g = np.zeros_like(self.weights)
            sum_b = 0.0
            for Xc, yc in client_data:
                for g_local, grad_b in self._local_update(Xc, yc):
                    pass
                sum_g += g_local
                sum_b += grad_b

            # aggregate
            K = len(client_data)
            sigma = self.noise_multiplier * self.clip_norm
            noise_g = np.random.normal(0, sigma, size=self.weights.shape)
            noise_b = np.random.normal(0, sigma)
            avg_g = (sum_g + noise_g) / K
            avg_b = (sum_b + noise_b) / K

            # global
            self.weights -= self.lr * avg_g
            self.bias   -= self.lr * avg_b

    def predict(self, X):
        return X.dot(self.weights) + self.bias

<h4> Chuẩn bị dữ liệu và huấn luyện mô hình </h4>

In [ ]:
# Split data among clients
n_clients = 5
client_indices = np.array_split(np.arange(len(X)), n_clients)
client_X = [X.iloc[idx].to_numpy(dtype=np.float64) for idx in client_indices]
client_y = [y_log[idx] for idx in client_indices]

# Set aside one client for evaluation
X_test_fede = client_X[-1]
y_test_fede = np.expm1(client_y[-1])  # Reverse log1p

# Train the model
fede_model = FederatedGaussianDP(
    clip_norm=1.0,
    noise_multiplier=0.5,
    lr=0.01,
    local_epochs=1,
    batch_size=32
)

fede_model.federated_train(
    client_data=list(zip(client_X[:-1], client_y[:-1])),
    rounds=10
)

<h4> Đánh giá mô hình </h4>

In [ ]:
# Predict on held-out client
y_pred_log_fede = fede_model.predict(X_test_fede)
y_pred_fede = np.expm1(y_pred_log_fede)

# Metric calculations
def manual_metrics(y_true, y_pred):
    n = len(y_true)
    p = X_test_fede.shape[1]
    errors = y_true - y_pred
    mse = np.mean(errors ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(errors))
    ss_res = np.sum(errors ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else r2
    return mse, rmse, mae, r2, adj_r2

In [ ]:
mse_fede, rmse_fede, mae_fede, r2_fede, adj_r2_fede = manual_metrics(y_test_fede, y_pred_fede)

print("Federated DP Model Evaluation")
print(f"MSE      = {mse_fede:.2e}")
print(f"RMSE     = {rmse_fede:.2f}")
print(f"MAE      = {mae_fede:.2f}")
print(f"R²       = {r2_fede:.3f}")
print(f"Adj R²   = {adj_r2_fede:.3f}")

<h4> Tính toán ngân sách riêng tư </h4>

In [ ]:
def compute_eps_gaussian(rounds, noise_multiplier, delta=1e-5):
    return np.sqrt(2 * rounds * np.log(1 / delta)) * noise_multiplier + \
           rounds * noise_multiplier * (np.exp(noise_multiplier) - 1)

In [ ]:
eps_total = compute_eps_gaussian(rounds=10, noise_multiplier=0.5, delta=1e-5)
print(f"\nEstimated privacy: ε_total ≈ {eps_total:.3f} with δ = 1e-5")